# NeuralHydrology – evaluatie, controle en export

Deze notebook volgt de bestaande projectstructuur van `neural_hydrology`:

- notebook: `notebooks/visualisatie_runs.ipynb`;
- notebookconfig: `notebooks/config_visualisatie_runs.yml`;
- omgevingsafhankelijke paden via `neural_hydrology.paths` en repo-root `.env`;
- geen catalogus-, schema- of volumenaam hardcoded in de notebook;
- evaluatieresultaten worden persistent hergebruikt;
- uurlijkse timestamps gebruiken `date + time_step`;
- `NSE_1h` wordt correct gebruikt;
- nullwaarden worden expliciet gediagnosticeerd;
- zware Dash/Plotly-output is verwijderd.


In [0]:
from pathlib import Path
import gc
import os
import pickle
import re
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
import yaml

from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
from neuralhydrology.evaluation import get_tester
from neuralhydrology.utils.config import Config

repo_root = Path.cwd()

if not (repo_root / "src" / "neural_hydrology").is_dir():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root / "src"))

from neural_hydrology.paths import get_path, get_project_root

warnings.simplefilter("ignore", FutureWarning)

## 1. Runconfig laden

In [0]:
PROJECT_CODE_ROOT = get_project_root()

RUNCONFIG_FILENAME = "config_visualisatie_runs.yml"

# Standaard volgens de repo-structuur: notebooks/config_visualisatie_runs.yml.
# Alleen voor uitzonderingen kan NH_VIS_RUNCONFIG een expliciet pad geven.
explicit_runconfig = os.environ.get("NH_VIS_RUNCONFIG", "").strip()

if explicit_runconfig:
    RUNCONFIG_PATH = Path(explicit_runconfig).expanduser()
    if not RUNCONFIG_PATH.is_absolute():
        RUNCONFIG_PATH = PROJECT_CODE_ROOT / RUNCONFIG_PATH
else:
    RUNCONFIG_PATH = PROJECT_CODE_ROOT / "notebooks" / RUNCONFIG_FILENAME

if not RUNCONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Visualisatieconfig niet gevonden: {RUNCONFIG_PATH}\n"
        f"Verwachte repo-root: {PROJECT_CODE_ROOT}\n"
        f"Plaats {RUNCONFIG_FILENAME} in de map notebooks/."
    )

with open(RUNCONFIG_PATH, "r", encoding="utf-8") as f:
    RUNCFG = yaml.safe_load(f)

RUNS_FOLDER = RUNCFG["runs"]["experiment_folder"]
RUN_NAMES = list(RUNCFG["runs"]["names"])

EVALUATION_DEVICE = RUNCFG["evaluation"].get("device", "cpu")
EVALUATE_IF_CACHE_MISSING = bool(
    RUNCFG["evaluation"].get("evaluate_if_cache_missing", True)
)
MODEL_SELECTION_FREQUENCY = RUNCFG["evaluation"].get(
    "model_selection_frequency", "1h"
)

EXPORT_YEAR = int(RUNCFG["export"].get("year", 2022))
EXPECTED_HOURS_PER_BASIN = int(
    RUNCFG["export"].get("expected_hours_per_basin", 8760)
)

print(f"Repo-root:    {PROJECT_CODE_ROOT}")
print(f"Config:       {RUNCONFIG_PATH}")
print(f"Experiment:   {RUNS_FOLDER}")
print(f"Runs:         {len(RUN_NAMES)}")
print(f"Exportjaar:   {EXPORT_YEAR}")


## 2. Projectpaden via bestaande `paths.py` / `.env`

De repo heeft al centrale padresolutie. De notebook gebruikt daarom dezelfde `DATA_DIR`, `OUTPUT_DIR` en `RETRAIN_BASE_DIR` als de rest van het project, in plaats van zelf door `/Volumes/...` te zoeken.


In [0]:
# Gebruik exact dezelfde padresolutie als de bestaande scripts/jobs.
DATA_ROOT = get_path("DATA_DIR")
OUTPUT_ROOT = get_path("OUTPUT_DIR")
RETRAIN_BASE_ROOT = get_path("RETRAIN_BASE_DIR")

RUNS_ROOT = RETRAIN_BASE_ROOT / RUNS_FOLDER
CACHE_ROOT = OUTPUT_ROOT / RUNCFG["evaluation"].get(
    "cache_subdir", "evaluation_cache"
)

# Bestaande projectconventies onder de trainingsdata.
ATTRIBUTES_PATH = DATA_ROOT / "attributes" / "polders_data_aangevuld.csv"
BASIN_FILE_PATH = DATA_ROOT / "hdsr_polders.txt"

for run in RUN_NAMES:
    run_cfg = RUNS_ROOT / run / "config.yml"
    if not run_cfg.exists():
        raise FileNotFoundError(
            f"Run niet gevonden via RETRAIN_BASE_DIR: {run_cfg}\n"
            "Controleer OUTPUT_DIR/RETRAIN_BASE_DIR in repo-root .env."
        )

for p, label in [
    (DATA_ROOT, "DATA_DIR"),
    (OUTPUT_ROOT, "OUTPUT_DIR"),
    (ATTRIBUTES_PATH, "attributes"),
    (BASIN_FILE_PATH, "basin file"),
]:
    if not p.exists():
        raise FileNotFoundError(
            f"{label} niet gevonden: {p}\n"
            "Controleer de repo-root .env."
        )

CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print("Projectpaden:")
print(f"  data          : {DATA_ROOT}")
print(f"  output        : {OUTPUT_ROOT}")
print(f"  retrain base  : {RETRAIN_BASE_ROOT}")
print(f"  runs          : {RUNS_ROOT}")
print(f"  cache         : {CACHE_ROOT}")


## 3. Beste epoch per run bepalen uit TensorBoard

In [0]:
def extract_tensorboard_scalars(logdir: Path):
    scalars = {}
    for root, _, files in os.walk(logdir):
        if any(f.startswith("events.out.tfevents") for f in files):
            event_acc = EventAccumulator(root)
            event_acc.Reload()
            for tag in event_acc.Tags().get("scalars", []):
                scalars[tag] = [(int(e.step), float(e.value)) for e in event_acc.Scalars(tag)]
    return scalars


def find_tag(data: dict, requested: str) -> str:
    requested_lower = requested.lower()
    for tag in data:
        if tag.lower() == requested_lower:
            return tag
    raise KeyError(f"TensorBoard-tag niet gevonden: {requested}. Beschikbaar: {list(data)}")


def select_best_epoch(run_folder: Path):
    data = extract_tensorboard_scalars(run_folder)
    ecfg = RUNCFG["evaluation"]["epoch_selection"]

    tag_1d = find_tag(data, ecfg.get("metric_1d", "valid/mean_nse_1d"))
    tag_1h = find_tag(data, ecfg.get("metric_1h", "valid/mean_nse_1h"))

    by_step_1d = dict(data[tag_1d])
    by_step_1h = dict(data[tag_1h])
    common_steps = sorted(set(by_step_1d) & set(by_step_1h))

    rows = []
    for step in common_steps:
        nse_1d = by_step_1d[step]
        nse_1h = by_step_1h[step]
        if np.isfinite(nse_1d) and np.isfinite(nse_1h):
            combined = (nse_1d + nse_1h) / 2.0
            rows.append((step, nse_1d, nse_1h, combined))

    if not rows:
        raise ValueError(f"Geen geldige gezamenlijke 1D/1h-validatiescores in {run_folder}")

    best = max(rows, key=lambda r: r[3])
    return {
        "epoch": int(best[0]),
        "nse_1d": float(best[1]),
        "nse_1h": float(best[2]),
        "combined": float(best[3]),
        "history": pd.DataFrame(rows, columns=["epoch", "NSE_1D", "NSE_1h", "combined"]),
    }


BEST_EPOCHS = {}
for run in RUN_NAMES:
    info = select_best_epoch(RUNS_ROOT / run)
    BEST_EPOCHS[run] = info["epoch"]
    print(
        f"{run}: epoch {info['epoch']} | "
        f"NSE 1D={info['nse_1d']:.3f} | "
        f"NSE 1h={info['nse_1h']:.3f} | "
        f"gemiddeld={info['combined']:.3f}"
    )


De oude notebook selecteerde het beste epoch alleen op `mean_nse_1d` en leidde het epoch af uit de positie in de lijst. Hier wordt de echte TensorBoard-step gebruikt en wordt, in lijn met de projectlogica, het gemiddelde van 1D- en 1h-NSE genomen. Er hoeven dus geen modelbestanden meer handmatig naar een backupmap te worden verplaatst.

## 4. Evaluatieresultaten laden uit cache of éénmalig berekenen

In [0]:
def safe_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))


def cache_path_for(run: str, epoch: int, period: str) -> Path:
    return CACHE_ROOT / f"{safe_name(run)}_epoch{epoch:03d}_{period}_results.pkl"


def native_result_path(run_folder: Path, epoch: int, period: str) -> Path:
    filename = "validation_results.p" if period == "validation" else f"{period}_results.p"
    return run_folder / period / f"model_epoch{epoch:03d}" / filename


def make_runtime_config(run_folder: Path) -> Config:
    with open(run_folder / "config.yml", "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    # Historische runconfigs mogen absolute paden bevatten. Voor deze notebook
    # worden de paden altijd naar het automatisch gevonden project omgeleid.
    cfg["data_dir"] = str(DATA_ROOT)

    for key in ("train_basin_file", "validation_basin_file", "test_basin_file"):
        if key in cfg:
            cfg[key] = str(BASIN_FILE_PATH)

    cfg["device"] = EVALUATION_DEVICE
    return Config(cfg)


results_list = []

for model_idx, run in enumerate(RUN_NAMES):
    epoch = BEST_EPOCHS[run]
    run_folder = RUNS_ROOT / run
    model_results = []

    print(f"\nModel {model_idx + 1}/{len(RUN_NAMES)}: {run} — epoch {epoch}")

    for period in RUNCFG["evaluation"].get("periods", ["validation", "test"]):
        cache_path = cache_path_for(run, epoch, period)

        native_path = native_result_path(run_folder, epoch, period)

        if cache_path.exists():
            print(f"[CACHE] {period}: {cache_path.name}")
            with open(cache_path, "rb") as f:
                result = pickle.load(f)

        elif native_path.exists():
            # NeuralHydrology bewaart evaluate(save_results=True) zelf ook persistent.
            # Gebruik dat resultaat voordat we iets opnieuw berekenen.
            print(f"[NH RESULT] {period}: {native_path}")
            with open(native_path, "rb") as f:
                result = pickle.load(f)

            tmp_path = cache_path.with_suffix(cache_path.suffix + ".tmp")
            with open(tmp_path, "wb") as f:
                pickle.dump(result, f, protocol=pickle.HIGHEST_PROTOCOL)
            tmp_path.replace(cache_path)
            print(f"[CACHE] gekopieerd naar: {cache_path}")

        else:
            if not EVALUATE_IF_CACHE_MISSING:
                raise FileNotFoundError(
                    f"Geen cache of NeuralHydrology-resultaat voor {run} / {period} / epoch {epoch}."
                )

            print(f"[EVALUATE] {period}: geen bestaand resultaat; éénmalig berekenen")
            tester_config = make_runtime_config(run_folder)
            tester = get_tester(
                tester_config,
                run_folder,
                init_model=True,
                period=period,
            )
            result = tester.evaluate(
                epoch=epoch,
                save_results=True,
                metrics=tester_config.metrics,
            )

            # Eerst tijdelijk schrijven en daarna atomair vervangen, zodat een
            # kernelcrash geen half picklebestand achterlaat.
            tmp_path = cache_path.with_suffix(cache_path.suffix + ".tmp")
            with open(tmp_path, "wb") as f:
                pickle.dump(result, f, protocol=pickle.HIGHEST_PROTOCOL)
            tmp_path.replace(cache_path)
            print(f"[CACHE] opgeslagen: {cache_path}")

            del tester
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        model_results.append(result)

    results_list.append(model_results)

print(f"\nBeschikbaar: {len(results_list)} modellen")


## 5. Eenheden, NSE en correcte tijdassen

In [0]:
df_attributes = pd.read_csv(ATTRIBUTES_PATH).set_index("SHAPE_ID")
series_keys = sorted(results_list[0][0].keys())


def to_discharge_m3_s(values, basin: str, clip_negative: bool = True):
    """afvoer [m/s] × oppervlakte [m²] = debiet [m³/s]. NaN blijft NaN."""
    if basin not in df_attributes.index:
        raise KeyError(f"{basin} ontbreekt in {ATTRIBUTES_PATH}")

    area = float(df_attributes.loc[basin, "oppervlak"])
    if not np.isfinite(area) or area <= 0:
        raise ValueError(f"Ongeldig oppervlak voor {basin}: {area}")

    result = np.asarray(values, dtype=float) * area
    if clip_negative:
        result = np.where(np.isfinite(result), np.maximum(result, 0.0), np.nan)
    return result


def get_nse(result_block: dict, frequency: str):
    key = f"NSE_{frequency}"
    value = result_block.get(key, np.nan)
    try:
        value = float(np.asarray(value).squeeze())
    except (TypeError, ValueError):
        return np.nan
    return value if np.isfinite(value) else np.nan


def _extract_date_timestep_variable(ds, variable: str, basin: str):
    da = ds[variable].squeeze(drop=True)
    if "date" not in da.dims or "time_step" not in da.dims:
        raise ValueError(f"{basin}: onverwachte dimensies voor {variable}: {da.dims}")
    return np.asarray(da.transpose("date", "time_step").values, dtype=float).reshape(-1)


def hourly_frame(result_block: dict, basin: str) -> pd.DataFrame:
    ds = result_block["xr"]
    if "date" not in ds.dims or "time_step" not in ds.dims:
        raise ValueError(f"{basin}: 1h-resultaat mist date/time_step: {ds.dims}")

    ds = ds.isel(time_step=slice(-24, None))

    time_steps = np.asarray(ds["time_step"].values)
    dates = pd.to_datetime(ds["date"].values)

    # NeuralHydrology gebruikt time_step normaal als integer offset
    # in eenheden van de betreffende frequentie, hier dus uren.
    if np.issubdtype(time_steps.dtype, np.integer):
        offsets = pd.to_timedelta(time_steps, unit="h")

    elif np.issubdtype(time_steps.dtype, np.timedelta64):
        # Voor compatibiliteit met eventuele andere/oude resultaten
        offsets = pd.to_timedelta(time_steps)

    else:
        raise TypeError(
            f"{basin}: onverwacht time_step-type: {time_steps.dtype}"
        )

    datetimes = (
        dates.values[:, None]
        + offsets.to_numpy()[None, :]
    ).reshape(-1)

    obs_raw = _extract_date_timestep_variable(ds, "afvoer_obs", basin)
    sim_raw = _extract_date_timestep_variable(ds, "afvoer_sim", basin)

    if not (len(datetimes) == len(obs_raw) == len(sim_raw)):
        raise ValueError(
            f"{basin}: lengteverschil datetime={len(datetimes)}, "
            f"obs={len(obs_raw)}, sim={len(sim_raw)}"
        )

    return pd.DataFrame({
        "datetime": pd.to_datetime(datetimes),
        "afvoer_obs_raw": obs_raw,
        "afvoer_sim_raw": sim_raw,
        "observed_m3_s": to_discharge_m3_s(obs_raw, basin),
        "simulated_m3_s": to_discharge_m3_s(sim_raw, basin, clip_negative=False),
    })


def daily_frame(result_block: dict, basin: str) -> pd.DataFrame:
    ds = result_block["xr"]

    def extract(variable: str):
        da = ds[variable]
        if "time_step" in da.dims:
            da = da.isel(time_step=-1)
        return np.asarray(da.squeeze(drop=True).values, dtype=float).reshape(-1)

    dates = pd.to_datetime(ds["date"].values)
    obs_raw = extract("afvoer_obs")
    sim_raw = extract("afvoer_sim")

    if not (len(dates) == len(obs_raw) == len(sim_raw)):
        raise ValueError(f"{basin}: onverwachte dagelijkse shapes")

    return pd.DataFrame({
        "datetime": dates,
        "observed_m3_s": to_discharge_m3_s(obs_raw, basin),
        "simulated_m3_s": to_discharge_m3_s(sim_raw, basin),
    })

print(f"Afvoergebieden in resultaten: {len(series_keys)}")
print(f"Afvoergebieden in attributentabel: {len(df_attributes)}")


## 6. NSE-overzicht

In [0]:
def metric_values(model_idx: int, dataset_idx: int, frequency: str):
    out = []
    for basin in series_keys:
        value = get_nse(results_list[model_idx][dataset_idx][basin][frequency], frequency)
        out.append(round(value, 3) if np.isfinite(value) else np.nan)
    return out


nse_validation = pd.DataFrame(index=series_keys)
nse_test = pd.DataFrame(index=series_keys)

for model_idx in range(len(RUN_NAMES)):
    nse_validation[f"model_{model_idx + 1}_NSE_1D"] = metric_values(model_idx, 0, "1D")
    nse_validation[f"model_{model_idx + 1}_NSE_1h"] = metric_values(model_idx, 0, "1h")
    nse_test[f"model_{model_idx + 1}_NSE_1D"] = metric_values(model_idx, 1, "1D")
    nse_test[f"model_{model_idx + 1}_NSE_1h"] = metric_values(model_idx, 1, "1h")

print("Validatie-NSE:")
display(nse_validation)
print("Test-NSE:")
display(nse_test)


NSE is dimensieloos. Het vermenigvuldigen van observatie en simulatie met dezelfde oppervlaktefactor verandert de NSE niet. Het achteraf afkappen van negatieve debieten op nul kan de score wel veranderen; daarom worden de opgeslagen NeuralHydrology-metrics hierboven ongewijzigd getoond.

## 7. Compacte tijdreeksvisualisatie

In [0]:
vcfg = RUNCFG.get("visualization", {})
selected_area = vcfg.get("default_area", series_keys[0])
dataset_name = str(vcfg.get("dataset", "test")).lower()
dataset_idx = 0 if dataset_name == "validation" else 1
start = pd.Timestamp(vcfg.get("start", f"{EXPORT_YEAR}-01-01"))
end = pd.Timestamp(vcfg.get("end", f"{EXPORT_YEAR}-02-01"))

if selected_area not in series_keys:
    raise KeyError(f"Onbekend afvoergebied in runconfig: {selected_area}")

frames = [
    hourly_frame(results_list[i][dataset_idx][selected_area]["1h"], selected_area)
    for i in range(len(RUN_NAMES))
]
frames = [df[(df["datetime"] >= start) & (df["datetime"] < end)].copy() for df in frames]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=frames[0]["datetime"],
    y=frames[0]["observed_m3_s"],
    mode="lines",
    name="Gemeten",
))

for model_idx, df in enumerate(frames):
    nse = get_nse(results_list[model_idx][dataset_idx][selected_area]["1h"], "1h")
    nse_label = f"{nse:.2f}" if np.isfinite(nse) else "NaN"
    fig.add_trace(go.Scatter(
        x=df["datetime"],
        y=df["simulated_m3_s"],
        mode="lines",
        name=f"Model {model_idx + 1} — NSE 1h: {nse_label}",
    ))

fig.update_layout(
    title=f"{selected_area} — {dataset_name} — {start.date()} t/m {(end - pd.Timedelta(days=1)).date()}",
    xaxis_title="Datum/tijd",
    yaxis_title="Debiet [m³/s]",
    hovermode="x unified",
)
fig.show()


## 8. Beste model kiezen en export

Bij onvolledige brondata blijft de volledige candidate-export beschikbaar. Daarnaast wordt tijdelijk een export gemaakt met alleen afvoergebieden met volledige dekking in het exportjaar.

In [0]:
frequency = MODEL_SELECTION_FREQUENCY
validation_scores = []

for model_idx in range(len(RUN_NAMES)):
    basin_scores = [
        get_nse(results_list[model_idx][0][basin][frequency], frequency)
        for basin in series_keys
    ]
    basin_scores = [x for x in basin_scores if np.isfinite(x)]
    score = float(np.mean(basin_scores)) if basin_scores else np.nan
    validation_scores.append(score)
    print(f"Model {model_idx + 1}: gemiddelde validatie-NSE {frequency} = {score:.3f}")

if not np.isfinite(validation_scores).any():
    raise ValueError(f"Geen geldige validatie-NSE gevonden voor {frequency}")

best_model_idx = int(np.nanargmax(validation_scores))
best_run = RUN_NAMES[best_model_idx]
print(f"\nBeste model: {best_model_idx + 1} — {best_run} — epoch {BEST_EPOCHS[best_run]}")

start_year = pd.Timestamp(f"{EXPORT_YEAR}-01-01 00:00:00")
end_year = pd.Timestamp(f"{EXPORT_YEAR + 1}-01-01 00:00:00")

test_results = results_list[best_model_idx][1]
export_frames = []
diagnostics = []

for basin in series_keys:
    h = hourly_frame(test_results[basin]["1h"], basin)
    h = h[(h["datetime"] >= start_year) & (h["datetime"] < end_year)].copy()

    diagnostics.append({
        "afvoergebied_id": basin,
        "rows": len(h),
        "obs_nulls": int(h["afvoer_obs_raw"].isna().sum()),
        "sim_nulls": int(h["afvoer_sim_raw"].isna().sum()),
        "duplicate_datetimes": int(h["datetime"].duplicated().sum()),
        "negative_sim_raw": int((h["afvoer_sim_raw"] < 0).sum()),
        "sim_min_raw": h["afvoer_sim_raw"].min(),
        "sim_max_raw": h["afvoer_sim_raw"].max(),
    })

    export_frames.append(
        h[["datetime", "simulated_m3_s"]]
        .rename(columns={"simulated_m3_s": "discharge_m3_s"})
        .assign(afvoergebied_id=basin)
    )

diagnostics_df = pd.DataFrame(diagnostics).sort_values(
    ["sim_nulls", "obs_nulls", "afvoergebied_id"],
    ascending=[False, False, True],
)

diagnostic_path = OUTPUT_ROOT / RUNCFG["export"]["diagnostics_filename"]
diagnostics_df.to_csv(diagnostic_path, index=False)
print(f"Diagnostiek: {diagnostic_path}")
display(diagnostics_df)

df_export = (
    pd.concat(export_frames, ignore_index=True)
    .sort_values(["afvoergebied_id", "datetime"])
    .reset_index(drop=True)
)

candidate_path = OUTPUT_ROOT / RUNCFG["export"]["candidate_filename"]
df_export.to_csv(candidate_path, index=False)
print(f"Candidate-export (ook bij problemen): {candidate_path}")

bad = diagnostics_df[
    (diagnostics_df["rows"] != EXPECTED_HOURS_PER_BASIN)
    | (diagnostics_df["sim_nulls"] > 0)
    | (diagnostics_df["duplicate_datetimes"] > 0)
]

print("\nExportcontrole")
print("----------------")
print(f"Rijen:              {len(df_export):,}")
print(f"Afvoergebieden:     {df_export['afvoergebied_id'].nunique()}")
print(f"Null voorspellingen:{df_export['discharge_m3_s'].isna().sum():,}")
print(f"Duplicaten:         {df_export.duplicated(['afvoergebied_id', 'datetime']).sum():,}")
print(f"Start:              {df_export['datetime'].min()}")
print(f"Einde:              {df_export['datetime'].max()}")

if bad.empty:
    final_path = OUTPUT_ROOT / RUNCFG["export"]["prediction_filename"]
    tmp_final = final_path.with_suffix(final_path.suffix + ".tmp")

    df_export.to_csv(tmp_final, index=False)
    tmp_final.replace(final_path)

    print(f"\nDefinitieve export: {final_path}")

else:
    print(
        "\nDefinitieve export NIET overschreven: "
        "onderstaande gebieden hebben een probleem."
    )
    display(bad)

    # Tijdelijke workaround:
    # exporteer alleen afvoergebieden met volledige dekking in het exportjaar.
    bad_basins = set(bad["afvoergebied_id"])

    complete_export = (
        df_export[
            ~df_export["afvoergebied_id"].isin(bad_basins)
        ]
        .copy()
        .sort_values(["afvoergebied_id", "datetime"])
        .reset_index(drop=True)
    )

    provisional_path = (
        OUTPUT_ROOT
        / f"ml_discharge_{EXPORT_YEAR}_complete_basins_only.csv"
    )

    complete_export.to_csv(provisional_path, index=False)

    print("\nTIJDELIJKE EXPORT")
    print("-----------------")
    print(f"Bestand:            {provisional_path}")
    print(f"Afvoergebieden:     {complete_export['afvoergebied_id'].nunique()}")
    print(f"Rijen:              {len(complete_export):,}")
    print(
        f"Null voorspellingen:"
        f"{complete_export['discharge_m3_s'].isna().sum():,}"
    )
    print(
        "Uitgesloten:        "
        + ", ".join(sorted(bad_basins))
    )
    print(
        "\nLET OP: dit is geen definitieve 40-gebieden-export. "
        "De uitgesloten gebieden moeten later worden hersteld."
    )


## 9. Controlevisualisatie van de export voor Rob

Onderstaande controles gebruiken de export die daadwerkelijk gedeeld kan worden. Als enkele afvoergebieden onvolledige brondata hebben, is dit dus de tijdelijke `complete_basins_only`-export; anders de volledige export.

In [0]:
rob_export = (
    complete_export.copy()
    if "complete_export" in globals()
    else df_export.copy()
)

rob_summary = (
    rob_export
    .groupby("afvoergebied_id")
    .agg(
        gemiddeld_debiet_m3_s=("discharge_m3_s", "mean"),
        mediaan_debiet_m3_s=("discharge_m3_s", "median"),
        maximum_debiet_m3_s=("discharge_m3_s", "max"),
        aantal_uren=("discharge_m3_s", "size"),
    )
)

rob_summary["jaarvolume_m3"] = (
    rob_export
    .groupby("afvoergebied_id")["discharge_m3_s"]
    .sum()
    * 3600
)

rob_summary = (
    rob_summary
    .sort_values("jaarvolume_m3", ascending=False)
    .reset_index()
)

print(
    f"Export voor visualisatie: {len(rob_export):,} rijen, "
    f"{rob_export['afvoergebied_id'].nunique()} afvoergebieden, "
    f"{rob_export['discharge_m3_s'].isna().sum():,} nulls."
)

display(rob_summary)


In [0]:
ranking = rob_summary.sort_values("jaarvolume_m3", ascending=True)

fig = go.Figure(
    go.Bar(
        x=ranking["jaarvolume_m3"] / 1_000_000,
        y=ranking["afvoergebied_id"],
        orientation="h",
    )
)

fig.update_layout(
    title=f"Voorspeld jaarvolume {EXPORT_YEAR} per afvoergebied",
    xaxis_title="Jaarvolume [miljoen m³]",
    yaxis_title="Afvoergebied",
    height=max(600, 24 * len(ranking)),
)

fig.show()


In [0]:
available_rob_basins = sorted(rob_export["afvoergebied_id"].unique())

selected_rob_area = RUNCFG.get("visualization", {}).get(
    "default_area",
    available_rob_basins[0],
)

if selected_rob_area not in available_rob_basins:
    selected_rob_area = available_rob_basins[0]

rob_daily = (
    rob_export[
        rob_export["afvoergebied_id"] == selected_rob_area
    ]
    .set_index("datetime")["discharge_m3_s"]
    .resample("1D")
    .mean()
)

fig = go.Figure(
    go.Scatter(
        x=rob_daily.index,
        y=rob_daily.values,
        mode="lines",
        name="Daggemiddeld debiet",
    )
)

fig.update_layout(
    title=f"{selected_rob_area} — voorspeld debiet in {EXPORT_YEAR}",
    xaxis_title="Datum",
    yaxis_title="Debiet [m³/s]",
    hovermode="x unified",
)

fig.show()


## 10. Databricks App synchroniseren en lokale app-data voorbereiden

De app-service-principal krijgt geen toegang tot de Unity Catalog Volume. Daarom wordt de deploybare app onder `/Workspace/Shared/neural_hydrology_dash_app` opgebouwd.

De map `/Workspace/Shared/neural_hydrology/app` is voortaan de **bron voor de app-code**. Bij de eerste run worden ontbrekende bronbestanden eenmalig uit de bestaande deploymap overgenomen. Daarna overschrijft iedere notebook-run `app.py`, `app.yaml`, `requirements.txt`, `README.md` en `assets/` vanuit de bronmap. Assets worden bewust bestand-voor-bestands gesynchroniseerd, omdat `shutil.copytree` op Databricks `/Workspace` niet altijd betrouwbaar werkt. De data onder `data/` wordt vervolgens opnieuw opgebouwd.


In [0]:
import shutil

APP_SOURCE_DIR = Path("/Workspace/Shared/neural_hydrology_dash_app")
APP_TEMPLATE_DIR = PROJECT_CODE_ROOT / "app"
APP_DATA_DIR = APP_SOURCE_DIR / "data"
APP_MODEL_DIR = APP_DATA_DIR / "models"

APP_SOURCE_DIR.mkdir(parents=True, exist_ok=True)
APP_TEMPLATE_DIR.mkdir(parents=True, exist_ok=True)

APP_FILES = [
    "app.py",
    "app.yaml",
    "requirements.txt",
    "README.md",
]


def _bootstrap_if_missing(name: str) -> None:
    """Neem bij de eerste run de bestaande deployversie over als bronbestand."""
    src = APP_TEMPLATE_DIR / name
    deployed = APP_SOURCE_DIR / name

    if src.exists():
        return

    if not deployed.exists():
        raise FileNotFoundError(
            f"App-bronbestand ontbreekt zowel in {APP_TEMPLATE_DIR} als in {APP_SOURCE_DIR}: {name}"
        )

    shutil.copy2(deployed, src)
    print(f"[BOOTSTRAP] {name} -> {src}")


def _copy_tree_files(src_root: Path, dst_root: Path, *, remove_stale: bool = False) -> int:
    """Kopieer een map file-by-file; dit werkt betrouwbaarder op Databricks /Workspace dan shutil.copytree."""
    dst_root.mkdir(parents=True, exist_ok=True)

    source_files = []
    if src_root.exists():
        source_files = [p for p in src_root.rglob("*") if p.is_file()]

    source_rel = {p.relative_to(src_root) for p in source_files}

    if remove_stale and dst_root.exists():
        for dst in [p for p in dst_root.rglob("*") if p.is_file()]:
            rel = dst.relative_to(dst_root)
            if rel not in source_rel:
                dst.unlink()
                print(f"[SYNC] verwijderd oud asset: {dst}")

    for src in source_files:
        rel = src.relative_to(src_root)
        dst = dst_root / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

    return len(source_files)


# Eenmalig bestaande deploybestanden als bron vastleggen als de repo-map nog leeg is.
for filename in APP_FILES:
    _bootstrap_if_missing(filename)

src_assets = APP_TEMPLATE_DIR / "assets"
dst_assets = APP_SOURCE_DIR / "assets"

# Bootstrap assets file-by-file. shutil.copytree geeft op /Workspace soms ENOENT bij nested workspace files.
src_assets.mkdir(parents=True, exist_ok=True)
if not any(p.is_file() for p in src_assets.rglob("*")) and dst_assets.exists():
    copied = _copy_tree_files(dst_assets, src_assets)
    print(f"[BOOTSTRAP] assets: {copied} bestand(en) -> {src_assets}")

# Bronmap is vanaf hier leidend: iedere run overschrijft de deploybare app-code.
for filename in APP_FILES:
    src = APP_TEMPLATE_DIR / filename
    dst = APP_SOURCE_DIR / filename
    shutil.copy2(src, dst)
    print(f"[SYNC] {src} -> {dst}")

# Assets ook file-by-file synchroniseren en oude deploy-assets verwijderen.
copied_assets = _copy_tree_files(src_assets, dst_assets, remove_stale=True)
print(f"[SYNC] assets: {copied_assets} bestand(en) -> {dst_assets}")

print(f"App-code gesynchroniseerd vanuit: {APP_TEMPLATE_DIR}")


In [0]:
APP_SOURCE_DIR = Path("/Workspace/Shared/neural_hydrology_dash_app")
APP_DATA_DIR = APP_SOURCE_DIR / "data"
APP_MODEL_DIR = APP_DATA_DIR / "models"

# Modelbestanden volledig opnieuw opbouwen zodat oude/stale afvoergebieden niet blijven staan.
if APP_MODEL_DIR.exists():
    shutil.rmtree(APP_MODEL_DIR)
APP_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 1. Export voor Rob als gzip-compressed CSV.
# De gewone CSV is ~15 MB; de .csv.gz blijft onder de 10 MB app-filelimiet.
rob_export_for_app = (
    complete_export.copy()
    if "complete_export" in globals()
    else df_export.dropna(subset=["discharge_m3_s"]).copy()
)

rob_app_path = APP_DATA_DIR / "rob_export_2022.csv.gz"

rob_export_for_app[
    ["datetime", "discharge_m3_s", "afvoergebied_id"]
].to_csv(
    rob_app_path,
    index=False,
    compression="gzip",
)

# 2. Modelvergelijking per afvoergebied opsplitsen.
# Daardoor blijft ieder app-bestand klein en hoeft de app slechts één gebied tegelijk te lezen.
periods = RUNCFG["evaluation"].get("periods", ["validation", "test"])
period_to_idx = {period: idx for idx, period in enumerate(periods)}

required_periods = {"validation", "test"}
missing_periods = required_periods - set(period_to_idx)

if missing_periods:
    raise ValueError(
        "Voor de app ontbreken evaluatieperioden: "
        + ", ".join(sorted(missing_periods))
    )

manifest_rows = []
nse_rows = []

for basin in series_keys:
    basin_frames = []

    for period in ["validation", "test"]:
        dataset_idx = period_to_idx[period]

        model_1 = hourly_frame(
            results_list[0][dataset_idx][basin]["1h"],
            basin,
        )[
            ["datetime", "observed_m3_s", "simulated_m3_s"]
        ].rename(
            columns={"simulated_m3_s": "model_1_m3_s"}
        )

        model_2 = hourly_frame(
            results_list[1][dataset_idx][basin]["1h"],
            basin,
        )[
            ["datetime", "simulated_m3_s"]
        ].rename(
            columns={"simulated_m3_s": "model_2_m3_s"}
        )

        frame = model_1.merge(
            model_2,
            on="datetime",
            how="outer",
            validate="one_to_one",
        )

        frame["dataset"] = period
        basin_frames.append(frame)

        for model_idx, model_name in [(0, "Model 1"), (1, "Model 2")]:
            nse_rows.append({
                "afvoergebied_id": basin,
                "dataset": period,
                "model": model_name,
                "nse_1h": get_nse(
                    results_list[model_idx][dataset_idx][basin]["1h"],
                    "1h",
                ),
            })

    basin_data = (
        pd.concat(basin_frames, ignore_index=True)
        .sort_values(["dataset", "datetime"])
        .reset_index(drop=True)
    )

    basin_path = APP_MODEL_DIR / f"{basin}.csv.gz"

    basin_data.to_csv(
        basin_path,
        index=False,
        compression="gzip",
    )

    manifest_rows.append({
        "afvoergebied_id": basin,
        "bestand": f"models/{basin}.csv.gz",
        "rijen": len(basin_data),
        "bestand_mb": basin_path.stat().st_size / (1024 ** 2),
    })

manifest_df = pd.DataFrame(manifest_rows).sort_values("afvoergebied_id")
nse_app_df = pd.DataFrame(nse_rows).sort_values(
    ["afvoergebied_id", "dataset", "model"]
)

manifest_path = APP_DATA_DIR / "manifest.csv"
nse_path = APP_DATA_DIR / "nse_1h.csv"

manifest_df.to_csv(manifest_path, index=False)
nse_app_df.to_csv(nse_path, index=False)

print("Databricks App-data gereed:")
print(f"  app-map      : {APP_SOURCE_DIR}")
print(f"  Rob-export   : {rob_app_path}")
print(f"  manifest     : {manifest_path}")
print(f"  NSE          : {nse_path}")
print(f"  modelbestanden: {len(manifest_df)}")

max_file = manifest_df.loc[manifest_df["bestand_mb"].idxmax()]
print(
    f"  grootste modelbestand: "
    f"{max_file['afvoergebied_id']} — "
    f"{max_file['bestand_mb']:.2f} MB"
)
print(
    f"  Rob-export grootte: "
    f"{rob_app_path.stat().st_size / (1024 ** 2):.2f} MB"
)

display(manifest_df)


## 11. Cumulatieve ML-afvoer voor één gebied

In [0]:
basin = RUNCFG.get("visualization", {}).get("default_area", series_keys[0])

d = (
    df_export[df_export["afvoergebied_id"] == basin]
    .sort_values("datetime")
    .copy()
)

if d.empty:
    raise ValueError(f"Geen exportdata voor {basin}")

if d["discharge_m3_s"].isna().any():
    print(f"{basin} bevat nullwaarden; cumulatieve curve wordt bewust niet gemaakt.")
else:
    # m³/s × 3600 s = volume per uur [m³]
    d["volume_m3"] = d["discharge_m3_s"] * 3600.0
    d["cumulative_m3"] = d["volume_m3"].cumsum()

    downsample = RUNCFG.get("visualization", {}).get("cumulative_downsample", "1D")
    plot_d = (
        d.set_index("datetime")[["cumulative_m3"]]
        .resample(downsample)
        .last()
        .dropna()
        .reset_index()
    )

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=plot_d["datetime"],
        y=plot_d["cumulative_m3"],
        mode="lines",
        name="ML voorspelling",
    ))
    fig.update_layout(
        title=f"Cumulatieve ML-afvoer — {basin} — {EXPORT_YEAR}",
        xaxis_title="Datum",
        yaxis_title="Cumulatief volume [m³]",
    )
    fig.show()

    print(f"Totaal voorspeld volume: {d['volume_m3'].sum():,.0f} m³")


## Herstart na een Databricks-kernelcrash

De zware evaluatieresultaten staan in `output/evaluation_cache`. Na een kernelrestart kun je de notebook opnieuw vanaf boven uitvoeren; bestaande cachebestanden worden geladen. Alleen een evaluatie die nog niet volledig was opgeslagen moet opnieuw. De cachebestandsnaam bevat ook het gekozen epoch, zodat resultaten van verschillende epochs niet door elkaar kunnen raken.

De app-code staat onder `/Workspace/Shared/neural_hydrology/app`; de deploymap `/Workspace/Shared/neural_hydrology_dash_app` wordt bij sectie 10 telkens opnieuw vanuit die bron gesynchroniseerd.


In [0]:
run_folder = RUNS_ROOT / RUN_NAMES[best_model_idx]

with open(run_folder / "config.yml", "r", encoding="utf-8") as f:
    model_cfg = yaml.safe_load(f)

{
    "model": model_cfg.get("model"),
    "head": model_cfg.get("head"),
    "output_activation": model_cfg.get("output_activation"),
    "target_variables": model_cfg.get("target_variables"),
    "loss": model_cfg.get("loss"),
}

In [0]:
from pathlib import Path
import yaml

for run in RUN_NAMES:
    path = RUNS_ROOT / run / "config.yml"
    with open(path, "r") as f:
        cfg = yaml.safe_load(f)

    print(f"\n=== {run} ===")
    for key, value in cfg.items():
        print(f"{key}: {value}")